# Bildverarbeitung Belegarbeit 3

Diese Übungseinheit befasst sich mit der Anwendung von Methoden des Maschinellen Lernens und der Mustererkennung in der Bildverarbeitung. Ziel ist die Implementierung und Evaluation verschiedener Klassifikationsverfahren am Beispiel des CIFAR-10 Datensatzes. Der Fokus liegt dabei auf dem Vergleich zwischen klassischer Merkmalsextraktion und modernen Deep-Learning-Ansätzen.

Die Bearbeitung gliedert sich in folgende Schwerpunkte:

1.  **Feature Engineering (Merkmalsextraktion)**:
    Konstruktion und Evaluation von Merkmalsvektoren zur Repräsentation von Bildinhalten. Die Identifikation diskriminativer Merkmale ist ein entscheidender Schritt für die Leistungsfähigkeit klassischer Klassifikatoren.
    Es sollen eigenständig verschiedene Deskriptoren (siehe z.B. `skimage.feature`) evaluiert und kombiniert werden.
    Als exemplarische Basis dienen in diesem Notebook:
    *   **HOG (Histogram of Oriented Gradients)** zur Erfassung von Form- und Kantenstrukturen.
    *   **Farb- und Luminanz-Histogramme** zur Nutzung von spektralen Eigenschaften.

2.  **Klassische Klassifikationsverfahren**:
    Training und Validierung etablierter Algorithmen auf Basis der extrahierten Merkmale:
    *   **K-Nearest Neighbors (KNN)**: Untersuchung instanzbasierter Klassifikation
    Es erfolgt eine Evaluation des Einflusses des Hyperparametern  $k$ (Nachbarschaftsgröße ) auf die Metriken Genauigkeit (Accuracy), Präzision (Precision) und Trefferquote (Recall).

3.  **Das Perzeptron**:
    Implementierung eines Perzeptrons als fundamentales Modell künstlicher neuronaler Netze. Demonstration der linearen Separierbarkeit und Visualisierung der Entscheidungsgrenzen.

4.  **Deep Learning mit PyTorch**:
    Einsatz tiefer neuronaler Netze für das "End-to-End Learning" auf Rohdaten unter Verwendung des PyTorch-Frameworks:
    *   **Multi-Layer Perceptron (MLP)**: Implementierung eines vollvernetzten neuronalen Netzes.
    *   **Convolutional Neural Network (CNN)**: Anwendung faltender neuronaler Netze zur automatisierten Merkmalsextraktion.

Abschließend sollen die Leistungsfähigkeiten der klassischen Ansätze mit denen der neuronalen Architekturen verglichen werden.

### Vorbereitung der Arbeitsumgebung

Für die Bearbeitung der Aufgaben, insbesondere im Bereich Deep Learning (Teil 4), werden die Bibliotheken **PyTorch** und **Torchvision** benötigt.

Bitte stellen Sie sicher, dass diese in Ihrer Python-Umgebung installiert sind. Falls Sie den Paketmanager `uv` verwenden, führen Sie bitte vorab folgenden Befehl in Ihrem Terminal aus:

```bash
uv add torch torchvision
```
----

## Teil 1: Feature Engineering und Datenvorverarbeitung

Die Qualität eines Klassifikationsmodells hängt maßgeblich von der Repräsentation der Eingabedaten ab. Rohdaten in Form von Pixelmatrizen sind für klassische Verfahren wie K-Nearest Neighbors oder Entscheidungsbäume oft ungeeignet, da sie invariant gegenüber Transformationen sind und semantische Informationen nur implizit enthalten.

In diesem Abschnitt sollen Sie daher geeignete **Merkmalsvektoren (Feature Vectors)** konstruieren, die den Bildinhalt kompakt und diskriminativ beschreiben.

**Ihre Aufgaben:**

1.  **Exploration:** Machen Sie sich mit dem CIFAR-10 Datensatz vertraut.
2.  **Merkmalsextraktion:** Implementieren Sie die Funktion `create_feature_vector_for_image`. Nutzen Sie hierfür Bibliotheken wie `skimage.feature`.
    *   Evaluieren Sie struktur-basierte Merkmale wie **HOG (Histogram of Oriented Gradients)**.
    *   Untersuchen Sie die Relevanz von Farbinformationen (z.B. Histogramme im HSV-Farbraum).
3.  **Vektorisierung:** Kombinieren Sie die extrahierten Merkmale zu einem eindimensionalen Vektor, der als Eingabe für die nachfolgenden Klassifikatoren dient.

*Hinweis: Ziel ist es, eine Merkmalskombination zu finden, die eine möglichst hohe Trennschärfe zwischen den 10 Klassen ermöglicht.*

In [ ]:
from PIL import Image
import numpy as np
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from skimage import feature
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from ipywidgets import IntSlider, interact, fixed

In [ ]:
train_data = CIFAR10("cifar10", download=True)
test_data = CIFAR10("cifar10", train=False, download=True)

In [ ]:
def show_images(images, titles=None):
    n = len(images)
    _, axes = plt.subplots(1, n, figsize=(n * 3, 3))
    if n == 1:
        axes = [axes]
    for i in range(n):
        axes[i].imshow(images[i])
        axes[i].axis('off')
        if titles:
            axes[i].set_title(titles[i])
    plt.show()

images = [train_data[i][0] for i in range(8)]
titles = [train_data.classes[train_data[i][1]] for i in range(8)]
show_images(images, titles)

In [ ]:
def predict(test_data: np.ndarray, test_features: np.ndarray, classifier: KNeighborsClassifier | DecisionTreeClassifier, test_idx: int = 0):
    """Predicts and visualizes the results for a given test index."""
    num_visuals = 10
    fig, ax = plt.subplots(1, num_visuals, figsize=(num_visuals * 2, 2))
    for i in range(num_visuals):
        idx = (test_idx + i) % len(test_features)
        test_sample = test_features[idx].reshape(1, -1)
        test_pred = classifier.predict(test_sample)

        img = np.array(test_data.data[idx])
        ax[i].imshow(img)
        ax[i].set_title(f"P: {test_data.classes[test_pred[0]]}\nT: {test_data.classes[test_data[idx][1]]}")
        ax[i].axis('off')

    plt.show()

def dataset_labels_to_features(data: np.ndarray, cache: bool = False, prefix: str = "train") -> tuple[np.ndarray, np.ndarray]:
    """Converts a dataset of images and labels to feature vectors and labels.

    Args:
        data (np.ndarray): Dataset of images and labels.
        cache (bool): Whether to cache the features and labels to disk.
        prefix (str): Prefix for the cache files.

    Returns:
        tuple: features and labels as numpy arrays.
    """
    features = []
    labels = []

    if cache:
        try:
            features = np.load(f"{prefix}_features.npy").tolist()
            labels = np.load(f"{prefix}_labels.npy").tolist()
            return np.array(features),  np.array(labels)
        except FileNotFoundError:
            pass

    for record in tqdm(data):
        image, label = record[0], record[1]
        feature_vector = create_feature_vector_for_image(image)
        features.append(feature_vector)
        labels.append(label)

    if cache:
        np.save(f"{prefix}_features.npy", np.array(features))
        np.save(f"{prefix}_labels.npy", np.array(labels))

    return np.array(features), np.array(labels)

def calculate_accuracy_precision_recall(y_true, y_pred) -> tuple[float, float, float]:
    """Calculates accuracy, precision, and recall.

    Args:
        y_true (array-like): True labels.
        y_pred (array-like): Predicted labels.

    Returns:
        tuple: accuracy, precision, recall
    """
    from sklearn.metrics import accuracy_score, precision_score, recall_score
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)


    return accuracy, precision, recall

def clean_cache(prefix: str = "train") -> None:
    import os
    try:
        os.remove(f"{prefix}_features.npy")
    except FileNotFoundError:
        pass
    try:
        os.remove(f"{prefix}_labels.npy")
    except FileNotFoundError:
        pass

In [ ]:
def create_feature_vector_for_image(image: np.ndarray) -> np.ndarray:
    """Creates a feature vector for a given image.

    Args:
        image (np.ndarray): Input image.
    Returns:
        np.ndarray: Vector combining extracted features.

    See also:
        - skimage features: https://scikit-image.org/docs/stable/api/skimage.feature.html
    """
    import cv2
    from skimage.color import rgb2hsv

    img_array = np.array(image)

    hog_features = feature.hog(
        img_array,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        channel_axis=-1
    )

    img_hsv = rgb2hsv(img_array)
    h_hist, _ = np.histogram(img_hsv[:, :, 0], bins=16, range=(0, 1))
    s_hist, _ = np.histogram(img_hsv[:, :, 1], bins=8, range=(0, 1))
    v_hist, _ = np.histogram(img_hsv[:, :, 2], bins=8, range=(0, 1))

    h_hist = h_hist.astype(np.float32) / (h_hist.sum() + 1e-6)
    s_hist = s_hist.astype(np.float32) / (s_hist.sum() + 1e-6)
    v_hist = v_hist.astype(np.float32) / (v_hist.sum() + 1e-6)

    feature_vector = np.concatenate((hog_features, h_hist, s_hist, v_hist))
    return feature_vector

In [ ]:

#clean_cache("cifar10_train")
#clean_cache("cifar10_test")

train_features, train_labels = dataset_labels_to_features(train_data, cache=True, prefix="cifar10_train")
test_features, test_labels = dataset_labels_to_features(test_data, cache=True, prefix="cifar10_test")

### Klassische Klassifikationsverfahren

Nachdem Sie geeignete Merkmale extrahiert haben, sollen nun klassische Klassifikationsverfahren angewendet werden. Den Anfang macht der **K-Nearest Neighbors (KNN)** Algorithmus.

**Ihre Aufgaben:**

1.  **Hyperparameter-Optimierung:** Der wichtigste Parameter des KNN ist die Anzahl der Nachbarn $k$. Finden Sie den optimalen Wert für $k$, der die beste Klassifikationsleistung auf dem Testdatensatz liefert.
2.  **Evaluation:** Trainieren Sie den KNN-Klassifikator für verschiedene Werte von $k$ (z.B. $k=1$ bis $k=20$).
3.  **Visualisierung:** Erstellen Sie einen Plot, der die Abhängigkeit der Metriken von $k$ darstellt:
    *   **X-Achse:** Anzahl der Nachbarn $k$
    *   **Y-Achse:** Accuracy, Precision und Recall
    
    Analysieren Sie den Plot: Bei welchem $k$ ist die Performance am besten? Gibt es ein Overfitting bei zu kleinen $k$ oder Underfitting bei zu großen $k$?

**Tipps:**

- Nutzen Sie die Implementierungen von `KNeighborsClassifier` aus `sklearn.neighbors` für den KNN-Algorithmus.
- die `fit`-Methode des Klassifikators erwartet als Eingabe ein 2D-Array der Form `(num_samples, num_features)`. Achten Sie darauf, Ihre Merkmalsvektoren entsprechend zu formatieren.
- die `predict`-Methode gibt die vorhergesagten Klassenlabels für die Eingabedaten zurück. In diesem Fall müssen wir die Testdaten `test_features` verwenden, um Vorhersagen zu treffen.

In [ ]:

results = []
for k in range(1, 21):
    knn_k = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn_k.fit(train_features, train_labels)
    knn_preds = knn_k.predict(test_features)
    acc, prec, rec = calculate_accuracy_precision_recall(test_labels, knn_preds)
    results.append((k, acc, prec, rec))
    print(f"k={k:2d}  Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}")


k_values = [x[0] for x in results]
accuracies = [x[1] for x in results]
precisions = [x[2] for x in results]
recalls = [x[3] for x in results]

plt.figure(figsize=(10, 6))
plt.plot(k_values, accuracies, marker='o')
plt.plot(k_values, precisions, marker='x')
plt.plot(k_values, recalls, marker='s')
plt.legend(['Accuracy', 'Precision', 'Recall'])
plt.title('KNN Accuracy vs. Number of Neighbors (k)')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Test Accuracy')
plt.xticks(k_values)
plt.grid(True)

In [ ]:
K = 1

knn = KNeighborsClassifier(n_neighbors=K, n_jobs=-1)
knn.fit(train_features, train_labels)

interact(predict, test_data=fixed(test_data), test_features=fixed(test_features), classifier=fixed(knn), test_idx=IntSlider(min=0, max=len(test_features)-1, step=1, value=0));

## Teil 2: Das Perzeptron

In diesem Abschnitt verlassen wir die klassischen statistischen Verfahren und wenden uns den Grundlagen der neuronalen Netze zu. Das **Perzeptron** ist das einfachste Modell eines künstlichen Neurons und bildet den Baustein für komplexere Architekturen.

Es handelt sich um einen Algorithmus für das überwachte Lernen binärer Klassifikatoren. Ein Perzeptron entscheidet, ob ein Input zu einer Klasse gehört oder nicht, indem es eine lineare Entscheidungsfunktion lernt.

**Ihre Aufgaben:**

Implementieren Sie die fehlenden Kernkomponenten in der Klasse `Perceptron`:

1.  **`activation_function(self, z)`**:
    Implementieren Sie die Aktivierungsfunktion (z.B. Heaviside-Sprungfunktion), die den gewichteten Input auf einen binären Output (0 oder 1) abbildet.
2.  **`predict(self, X)`**:
    Berechnen Sie den Netto-Input (Linearkombination aus Inputs und Gewichten plus Bias) und wenden Sie die Aktivierungsfunktion an, um die Vorhersage zu generieren.
3.  **`fit(self, X, y)`**:
    Implementieren Sie die **Perzeptron-Lernregel**. Iterieren Sie über die Trainingsdaten und passen Sie die Gewichte und den Bias an, falls eine Fehlklassifikation vorliegt:
    $$ w_{neu} = w_{alt} + \eta \cdot (y_{soll} - y_{ist}) \cdot x $$
    $$ b_{neu} = b_{alt} + \eta \cdot (y_{soll} - y_{ist}) $$
    (wobei $\eta$ die Lernrate ist).

Anschließend wird das Perzeptron auf einem synthetischen, linear separierbaren Datensatz trainiert und die Entscheidungsgrenze visualisiert.

**Tipps:**
- Nutzen Sie `np.dot` für die Vektor-Matrix-Multiplikation zwischen Eingabedaten und Gewichten.
- Achten Sie darauf, die Gewichte und den Bias korrekt zu initialisieren und zu aktualisieren.
- Die Gewichtsmatrix sollte die Form `(num_features,)` haben, um mit den Eingabedaten multipliziert werden zu können.
- Der Bias ist ein Skalarwert, der zu jedem Netto-Input addiert wird.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

class Perceptron:
    def __init__(self, learning_rate:float=0.01, n_iterations:int=20):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None

    def activation_function(self, z:np.ndarray) -> np.ndarray:
        """Heaviside Step Function: 1 wenn z >= 0, sonst 0

        Args:
            z (np.ndarray): Input array.

        Returns:
            np.ndarray: Binary output array after applying the Heaviside step function.

        Hints:
            - np.where is useful for element-wise conditional operations
            - Heaviside Step Function: 1 if z >= 0, else 0
        """
        return np.where(z >= 0, 1, 0)

    def predict(self, X:np.ndarray) -> np.ndarray:
        """Predicts the class labels for given input data.

        Args:
            X (np.ndarray): Input data.

        Returns:
            np.ndarray: Predicted class labels.

        Hints:
            - Use np.dot for vectorized computation between input X and weights
            - Dont forget to add the bias term
            - Dont forget to apply the activation function
        """
        z = np.dot(X, self.weights) + self.bias
        return self.activation_function(z)

    def fit(self, X:np.ndarray, y:np.ndarray) -> None:
        """Trains the Perceptron model using the provided training data.

        Args:
            X (np.ndarray): Training input data.
            y (np.ndarray): Target labels.

        Hints:
            - Initialize weights and bias
            - Loop over the number of iterations
            - For each sample, make a prediction and update weights and bias based on the error
        """
        _, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iterations):
            for xi, yi in zip(X, y):
                y_pred = self.predict(xi.reshape(1, -1))[0]
                error = yi - y_pred
                self.weights += self.lr * error * xi
                self.bias += self.lr * error


In [ ]:
X, y = make_blobs(n_samples=100, centers=2, n_features=2, center_box=(-5, 2), random_state=42)
p = Perceptron(learning_rate=0.01, n_iterations=10)
p.fit(X, y)


print("Training abgeschlossen.")
print(f"Gelehrte Gewichte: {p.weights}")
print(f"Gelehrter Bias: {p.bias}")

fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(X[y==0, 0], X[y==0, 1], color='red', edgecolors='k', label='Klasse 0')
ax.scatter(X[y==1, 0], X[y==1, 1], color='blue', edgecolors='k', label='Klasse 1')

if p.weights is not None and np.sum(p.weights) != 0:
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x1_vals = np.linspace(x1_min, x1_max, 100)
    x2_vals = -(p.weights[0] * x1_vals + p.bias) / p.weights[1]
    ax.plot(x1_vals, x2_vals, 'k-', linewidth=2, label='Decision Boundary')

ax.set_title("Perzeptron Klassifikation")
ax.legend()
plt.show()

## Teil 3: Deep Learning mit PyTorch

Im letzten Teil der Übung nutzen wir moderne Deep-Learning-Methoden. Im Gegensatz zu den vorherigen Ansätzen, bei denen wir manuell Merkmale extrahiert haben (Feature Engineering), lernen tiefe neuronale Netze diese Merkmale direkt aus den Rohdaten (End-to-End Learning).

Wir verwenden hierfür das Framework **PyTorch**.

**Ihre Aufgaben:**

1.  **Multi-Layer Perceptron (MLP):**
    Implementieren Sie ein vollvernetztes neuronales Netz (Feed-Forward Network).
    In PyTorch erstellen Sie ein Modell, indem Sie von `torch.nn.Module` erben:
    *   Definieren Sie im Konstruktor `__init__` die Schichten (z.B. `torch.nn.Linear`).
    *   Es werden die Rohpixelwerte der Bilder als Eingabe verwendet (z.B. für CIFAR-10: 32x32x3 = 3072 Eingabeneuronen).
    *   Implementieren Sie die `forward`-Methode, die den Datenfluss durch das Netzwerk beschreibt (inkl. Aktivierungsfunktionen wie `torch.nn.ReLU`).

    *Experimentieren Sie:* Wie viele Hidden Layers und wie viele Neuronen pro Layer sind nötig, um eine gute Performance zu erreichen?

2.  **Convolutional Neural Network (CNN):**
    Implementieren Sie ein einfaches CNN, das speziell für Bilddaten geeignet ist. Nutzen Sie Convolutional Layers (`torch.nn.Conv2d`) zur Merkmalsextraktion und Pooling Layers (`torch.nn.MaxPool2d`) zur Dimensionsreduktion.
    
    Auch hier haben Sie freie Hand bei der Gestaltung der Architektur. Orientieren Sie sich gerne an der [offiziellen PyTorch Dokumentation](https://pytorch.org/docs/stable/nn.html).


**Tipp:**

- Variiere die Layer und deren Neuronenzahlen, um die beste Performance zu erzielen.
- Ändere die Anzahl der Trainings-Epochen, um Overfitting oder Underfitting zu vermeiden.
- Ändere die Lernrate des Optimierers, um die Konvergenz zu beeinflussen.

Vergleichen Sie am Ende die Ergebnisse des MLP und CNN mit den klassischen Ansätzen aus Teil 1.

In [ ]:
import torch
import torchvision.transforms as transforms

transform = transforms.Compose(
    [transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))]
)

train_data = CIFAR10("cifar10", download=True, transform=transform)
test_data = CIFAR10("cifar10", train=False, download=True, transform=transform)

training_loader = torch.utils.data.DataLoader(train_data, batch_size=16, shuffle=True)
validation_loader = torch.utils.data.DataLoader(test_data, batch_size=16, shuffle=False)

In [ ]:
class MlpClassifier(torch.nn.Module):
    def __init__(self, input_size: int, num_classes: int):
        super().__init__()
        self._name = "SimpleMLP"

        self.fc1 = torch.nn.Linear(input_size, 512)
        self.fc2 = torch.nn.Linear(512, 256)
        self.fc3 = torch.nn.Linear(256, 128)
        self.fc4 = torch.nn.Linear(128, num_classes)
        self.relu = torch.nn.ReLU()
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass of the MLP.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor after passing through the network.

        Hints:
            - Dont forget to flatten the input
        """
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

    def save(self):
        """Saves the model state to a file."""
        torch.save(self.state_dict(), f"{self._name}_model.pth")

    def load(self):
        """Loads the model state from a file."""
        self.load_state_dict(torch.load(f"{self._name}_model.pth"))

class SimpleCnnClassifier(torch.nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self._name = "SimpleCNN"

        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(3, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(32, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2),           
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2),           
            torch.nn.Conv2d(64, 128, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2),           
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(128 * 4 * 4, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.4),
            torch.nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass of the CNN.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor after passing through the network.
        """
        x = self.features(x)
        x = x.view(x.size(0), -1)  
        x = self.classifier(x)
        return x

    def save(self):
        """Saves the model state to a file."""
        torch.save(self.state_dict(), f"{self._name}_model.pth")

    def load(self):
        """Loads the model state from a file."""
        self.load_state_dict(torch.load(f"{self._name}_model.pth"))

class Trainer:
    """Simple Trainer class for training and evaluating PyTorch models.

    Args:
        model (torch.nn.Module): The PyTorch model to be trained and evaluated.
        learning_rate (float): Learning rate for the optimizer.
        device (str): Device to run the model on ('cpu' or 'cuda').
    """
    def __init__(self, model: torch.nn.Module, learning_rate: float = 0.001, device: str = 'cpu') -> None:
        self.model = model
        self.loss_fn = torch.nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        self.device = device
        self.model.to(self.device)

    def fit(self, train_loader: torch.utils.data.DataLoader, epochs: int):
        """Trains the model using the provided training data.

        Args:
            train_loader (torch.utils.data.DataLoader): DataLoader for training data.
            epochs (int): Number of training epochs.
        """
        self.model.train()
        for epoch in tqdm(range(epochs), desc="Training Epochs"):
            total_loss = 0
            for images, labels in tqdm(train_loader, desc="Training Batches", leave=False):
                self.optimizer.zero_grad()
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images)
                loss = self.loss_fn(outputs, labels)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_loader)
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

    
            self.model.save()

    def evaluate(self, validation_loader: torch.utils.data.DataLoader) -> tuple[np.ndarray, np.ndarray]:
        """Evaluates the model on the validation data.

        Args:
            validation_loader (torch.utils.data.DataLoader): DataLoader for validation data.

        Returns:
            Tuple[np.ndarray, np.ndarray]: Tuple containing predictions and ground truth labels.
        """
        self.model.eval()
        predictions = []
        gts = []
        with torch.no_grad():
            for images, labels in tqdm(validation_loader, desc="Validation Batches", leave=False):
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images)
                _, predicted = torch.max(outputs.data, 1)
                predictions.append(predicted.cpu().numpy())
                gts.append(labels.cpu().numpy())
        return np.concatenate(predictions), np.concatenate(gts)

Training des MLP Modells

*Für das Training, fit, nur dann ausführen, wenn du noch kein trainiertes Modell hast oder ein neues trainieren möchtest.
Möchtest du nur evaluieren, überspringe diesen Schritt.*

In [ ]:
model = MlpClassifier(32*32*3, 10)
trainer = Trainer(model, device="cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
trainer.fit(training_loader, epochs=5)

Evaluation des MLP Modells

Wenn du bereits ein model trainiert hast, kannst du es hier evaluieren. 
Der letzte checkpoint des trainierten Modells wird automatisch geladen und auf dem Testdatensatz evaluiert.

In [ ]:
model.load()
preds, gts = trainer.evaluate(validation_loader)
acc, precision, recall = calculate_accuracy_precision_recall(gts, preds)
print(f"MLP Test Accuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

Training des CNN Modells

*Für das Training, fit, nur dann ausführen, wenn du noch kein trainiertes Modell hast oder ein neues trainieren möchtest.
Möchtest du nur evaluieren, überspringe diesen Schritt.*

In [ ]:
model = SimpleCnnClassifier(10)
trainer = Trainer(model, device="cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
trainer.fit(training_loader, epochs=20)

Evaluation des CNN Models

In [ ]:

preds, gts = trainer.evaluate(validation_loader)
acc, precision, recall = calculate_accuracy_precision_recall(gts, preds)
print(f"CNN Test Accuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")